# Model proposal for continous integration

<a id="table"></a>

## Table of Contents

-  [1 Environment and Dependency Specification]

    - [1.1 Supported Python and Operating System]
    - [1.2 Installation via virtualenv (Recommended)]
    - [1.3 Key Python Dependencies]
    - [1.4 System-Level Dependencies]
    - [1.5 Alternative: pip + venv (no virtualenv)]


-  [2 Environment and Dependency Specification]

    - [2.1 Supported Operating Systems]
    - [2.2 Required System Libraries]
    - [2.3 File Descriptor Limits]
    - [2.4 Hardware Requirements]
    - [2.5 Storage Layout]

-  [3 Data Locations, Paths, and Access]

    - [3.1 External Data Source: Swisstopo STAC API]
    - [3.2 Reference Grid and Forest Mask]
    - [3.3 Key Input Data Files]
    - [3.4 Output Data Files]
    - [3.5 Temporary Files and Cleanup]
    - [3.6 Configuring Paths for a New Machine]


-  [4 Hardware and Storage Requirements]

    - [4.1 Dataset Size Estimates]
    - [4.2 RAM Requirements by Processing Stage]
    - [4.3 Runtime Benchmarks]

-  [5 End-to-End Quick Start]

    - [5.1 Continuous Update Pipeline (0_1_run_pipeline.sh)]
    - [5.2 Historical Backfill Pipeline (0_1_run_historic_analysis.sh)]
    - [5.3 Checking the Start Date Automatically]
    - [5.4 Running for a Single Date or Date Range]

-  [6 Entry Points for Common Use Cases]

    - [6.1 Running the Full Pipeline from Scratch]
    - [6.2 Updating the Historical NDVI Dataset (Incremental)]
    - [6.3 Generating TIFFs for a Set of Dates]
    - [6.4 Inspecting the Historical Zarr Structure]
    - [6.5 Plotting a Time Series for a Specific Location]
    - [6.6 Checking for Errors and Retrying Failed Downloads]

-  [7 Configuration and Hard-Coded Parameters]

    - [7.1 Date Parameters]
    - [7.2 Parallelism and Chunking Parameters]
    - [7.3 Sentinel-2 Processing Parameters]
    - [7.4 Outlier Detection Parameters]
    - [7.5 Compression Parameters]


-  [8 Data Format and Schema Documentation]

    - [8.1 Historical NDVI Zarr (ndvi_historic_vN.zarr)]
    - [8.2 mask_array Legend]
    - [8.3 Raw Download Zarr (tmp_*_downloaded_*.zarr)]
    - [8.4 COG GeoTIFF Output]
    - [8.5 Lookup Table Zarr (lookup_table_median_ndvi_v7.zarr)]

-  [9 Processing Methods]

    - [9.1 Overview and Product Levels]
    - [9.2 Seasonal Reference: The Median NDVI Model]
    - [9.3 Outlier Detection]
    - [9.4 LOESS Smoothing]
    - [9.5 Gap-filling by Linear Interpolation (L1)]
    - [9.6 Processing Latency and L2 Finalization]


-  [10 Validation and Error Handling]

    - [10.1 Built-in Validation in Script 5]
    - [10.2 Common Errors and Solutions]
    - [10.3 Versioning and Reproducibility]

-  [11 Script-by-Script Reference]

    - [11.1 0_2_get_last_date.py]
    - [11.2 1_extract_swisstopo_dataset.py / 1_download_satellite_images.py]
    - [11.3 1b_merge_satellite_image_downloads.py]
    - [11.4 4_merge_zarr.py]
    - [11.5 5_analyse_demo_efficient.py]
    - [11.6 6_create_cogtiff.py]
    - [11.7 4_plot_historic_tiff.py]
    - [11.8 2_historical_ndvi_test.py]


- [Case tested](#case-tested)

    - [Case 1: lowland broadleaf](#lowland-broadleaf)
    - [Case 2: highland broadleaf](#highland-broadleaf)
    - [Case 3: lowland evergreen](#lowland-evergreen)
    - [Case 4: highland broadleaf](#highland-evergreen)
    - [Case 5: fire-affected area](#fire)
    - [Case 6: nearby fire-affected area](#non-fire)
    - [Case 7: 2018 drought-affected area](#drought)
    - [Case 8: Vaia storm-affected area](#storm)




# 1. Environment and Dependency Specification

This chapter describes how to set up a working Python environment to run the Swiss Forest NDVI processing pipeline. The workflow has been developed and tested on Ubuntu 24.04 LTS and uses a virtualenv-based Python environment.

## 1.1 Supported Python and Operating System

The pipeline was developed and validated on:

-	Operating System: Ubuntu 24.04 LTS (Noble Numbat), kernel 6.8.0-90-generic
-	Python: 3.10+ (recommended 3.11)
-	Windows: not natively tested; Windows Subsystem for Linux (WSL2 with Ubuntu 22.04 or 24.04) is the recommended path for Windows users
-	macOS: not tested

## 1.2 Installation via virtualenv (Recommended)
The primary installation method is a Python virtual environment managed with virtualenv. The setup script 0_0_setup.sh documents all required steps:

``` bash
# 1. Clone the repository
git clone https://github.com/geco-bern/swiss-ndvi-processing.git
cd swiss-ndvi-processing

# 2. Install virtualenv if not already present
sudo apt install python3-virtualenv

# 3. Create the virtual environment
virtualenv .venv

# 4. Activate the environment
source .venv/bin/activate

# 5. Install Python dependencies
pip install -r requirements.txt
``` 

## 1.3 Key Python Dependencies
The following packages are central to the pipeline. All versions are pinned in requirements.txt:

| Package | Purpose | Notes |
| :--- | :--- | :--- |
xarray | Multi-dimensional labelled arrays |	Core data structure throughout |
zarr (v3)	| Chunked compressed array storage |	Zarr v3 format used |
dask / dask.distributed |	Parallel and out-of-core computation	| LocalCluster for parallelism |
rasterio |	Raster I/O and coordinate transforms |	Reads Swisstopo GeoTIFFs |
rioxarray |	Rasterio extension for xarray |	CRS and COG output |
pystac_client |	STAC API client for Swisstopo catalog |	Queries ch.swisstopo.swisseo_s2-sr_v100 |
statsmodels |	LOESS smoothing	| sm.nonparametric.lowess
numpy |	Numerical array operations |	Core dependency |
pandas |	Tabular data and date handling |	DatetimeIndex, date_range |
numcodecs / zarr.codecs	| Compression codecs |	Blosc/zstd used throughout |
affine |	Affine coordinate transformations	| Pixel-to-coordinate mapping |
tqdm	| Progress bars for download loops |	Loop progress in script 1

## 1.4 System-Level Dependencies
The following system libraries must be present before running pip install:
-	GDAL (>= 3.4): required by rasterio. Install with: sudo apt install gdal-bin libgdal-dev
-	PROJ (>= 8.0): coordinate reference system transformations. Install with: sudo apt install libproj-dev
-	libspatialindex: spatial indexing. Install with: sudo apt install libspatialindex-dev
-	Build tools: sudo apt install build-essential python3-dev

<div style="background-color: #ebf3fb; border: 1px solid #336699; padding: 10px; font-style: italic; color: #2c3e50;">
  If pip install fails for rasterio or fiona, ensure GDAL development headers are installed and that the GDAL version matches the rasterio version pinned in requirements.txt. Use: gdal-config --version to verify.
</div>

## 1.5 Alternative: pip + venv (no virtualenv)

``` bash

python3 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
pip install -r requirements.txt
``` 

## 1.6 Activating the Environment
All scripts assume the environment is activated before execution. The pipeline shell scripts activate it automatically via:

``` bash

VENV_PATH="/home/Shared/UniBe-swiss-ndvi/GitHub/swiss-ndvi-processing/.venv"
source "$VENV_PATH/bin/activate"
``` 

For interactive work, activate manually before launching Python or Jupyter:

``` bash

source /path/to/swiss-ndvi-processing/.venv/bin/activate
``` 


# 2. System-Level Dependencies and OS Support

This chapter documents hardware, operating system, and system library requirements in detail. It is intended for system administrators and users setting up the pipeline on new machines.

## 2.1 Supported Operating Systems


| OS | Support Level | Notes |
| :--- | :--- | :--- |
| **Ubuntu 24.04 LTS** | Primary / Tested | Main development and production environment. |
| **Ubuntu 22.04 LTS** | Supported | Minor path differences possible. |
| **Other Debian/Ubuntu** | Likely works | Install equivalent system packages. |
| **WSL2 (Ubuntu 22/24)** | Recommended for Windows | Use WSL2, not WSL1; mount data drives under `/mnt/`. |
| **macOS** | Not tested | Conda environment may work; GDAL install differs. |
| **Windows (native)** | Not supported | Use WSL2 instead. |

## 2.2 Required System Libraries
Install all system dependencies in a single command on Ubuntu/Debian:

``` bash

sudo apt update && sudo apt install -y \
  gdal-bin libgdal-dev \
  libproj-dev proj-data proj-bin \
  libspatialindex-dev \
  build-essential python3-dev python3-virtualenv \
  libhdf5-dev libnetcdf-dev
``` 

<div style="background-color: #ebf3fb; border: 1px solid #336699; padding: 10px; font-style: italic; color: #2c3e50;">
  libhdf5-dev and libnetcdf-dev are optional but recommended if h5netcdf or scipy-based NetCDF backends are needed.
</div>

## 2.3 File Descriptor Limits

The pipeline opens many Zarr chunk files simultaneously. The default OS limit of 1024 open file descriptors is insufficient for large datasets. Increase it before running:

``` bash

# Check current limit
ulimit -n

# Raise for the current session (historic analysis script does this automatically)
ulimit -n 8192
``` 

The shell script 0_1_run_historic_analysis.sh sets this automatically. For production deployments, set permanently in /etc/security/limits.conf:

``` bash

*  soft  nofile  8192
*  hard  nofile  65536
``` 
## 2.4 Hardware Requirements
The hardware requirements vary significantly depending on the spatial extent of the dataset (number of pixels) being processed. The codebase was developed and benchmarked on a multi-core workstation (referred to as 'tunder').

| Component | Minimum (demo, ~4K px) | Recommended (full CH, ~105M px) |
| :--- | :--- | :--- |
| **CPU cores** | 4 | 60–120 (Dask workers) |
| **RAM** | 32 GB | 500 GB – 1.5 TB (120 GB/worker x 10) |
| **Disk (data)** | 5 GB | 600 GB – 1.5 TB (Zarr stores + TIFFs) |
| **Disk (temp)** | 2 GB | 400 GB (intermediate Zarr during processing) |
| **Network** | 10 Mbit/s | 100 Mbit/s+ (Swisstopo download ~several GB/year) |



<div style="background-color: #ebf3fb; border: 1px solid #336699; padding: 10px; font-style: italic; color: #2c3e50;">
  The Dask dashboard (accessible at localhost:8343 during script 4 and 5 execution) provides real-time monitoring of memory and CPU usage. Adjust N_WORKERS and MEMORY_PER_WORKER in the scripts if your workstation has different resources.
</div>

## 2.5 Storage Layout

| Path | Purpose |
| :--- | :--- |
| `/mnt/data1/UniBe-swiss-ndvi/` | Primary data store (input/output data, TIFFs) |
| `/mnt/data2/UniBe-swiss-ndvi/` | Scratch space for temporary Zarr files and Dask spill |
| `/home/Shared/UniBe-swiss-ndvi/GitHub/` | Code repository and virtual environment |
| `/mnt/data1/UniBe-swiss-ndvi/backup/` | Backup copies of historical NDVI Zarr |


# 3. Data Locations, Paths, and Access

This chapter documents all data sources, file paths, and access methods required to run the pipeline. All hard-coded paths in the scripts follow a consistent naming convention.

## 3.1 External Data Source: Swisstopo STAC API

Satellite imagery is downloaded at runtime from the Swisstopo STAC API. No API key or authentication is required — the API is publicly accessible:

-	API endpoint: https://data.geo.admin.ch/api/stac/v0.9/
-	Collection: ch.swisstopo.swisseo_s2-sr_v100 (Sentinel-2 surface reflectance, 10m resolution)
-	Spatial coverage: Switzerland bounding box in WGS84 (lon: 5.70–10.60, lat: 45.80–47.95)
-	Temporal coverage: April 2017 to present
-	Overpass frequency: approximately every 3–5 days depending on orbit

The API returns asset links to cloud-hosted GeoTIFFs (bands-10m.tif, bands-20m.tif, masks-10m.tif). These are read directly with rasterio without downloading the full files.

## 3.2 Reference Grid and Forest Mask

A fixed spatial reference grid defines which pixels are processed. This was determined once from the Swisstopo VHI forest mask and is now hardcoded and stored as a compressed Zarr file:

| Parameter | Value |
| :--- | :--- |
| **Coordinate Reference System** | `EPSG:2056` (CH1903+ / LV95) |
| **Bounding box (L, B, R, T)** | `2474090`, `1065110`, `2851370`, `1310530` [m] |
| **Pixel resolution** | 10 m x 10 m |
| **Full grid size** | 24,542 rows x 37,728 cols = 925,920,576 pixels |
| **Forest pixels** | 105,715,396 pixels (IDs 0 to 105,715,395) |
| **Forest mask file** | `workflow_implementation/data/forest_mask_bits.zarr` |

The forest mask file must be present at the path referenced in script 1. It defines the pixel ID mapping used by all subsequent scripts.


## 3.3 Key Input Data Files

| File / Path | Description | Used by Scripts |
| :--- | :--- | :--- |
| `ndvi_historic_vN.zarr` | Historical processed NDVI time series (2017–present). The main evolving dataset. | 4, 5, 6 |
| `lookup_table_median_ndvi_v7.zarr` | Pre-computed median NDVI per pixel and day-of-year from the double-logistic seasonal model. | 5, 2_historical |
| `forest_mask_bits.zarr` | Packed-bit forest mask defining the 105M pixel domain. | 1 |
| `tmp_*_ndvi_01_downloaded_*.zarr` | **Temporary:** raw downloaded NDVI/NDSI per observation date. | Created by 1, read by 4 |
| `tmp_*_processed.zarr` | **Temporary:** daily resampled and merged NDVI ready for analysis. | Created by 4, read by 5 |

## 3.4 Output Data Files

| File / Path | Description |
| :--- | :--- |
| `data/tiffs/YYYYMMDD.tiff` | Cloud-optimised GeoTIFF of processed NDVI for a given date (COG, int16, EPSG:2056, deflate) |
| `data/tiffs/YYYYMMDD_mask.tiff` | Cloud-optimised GeoTIFF of the mask_array for the same date |
| `data/tiffs_historic_vN/` | Historic TIFF archive — one NDVI + mask TIFF pair per date |
| `ndvi_historic_vN.zarr` (updated) | The historical Zarr is updated in-place (appended) by script 5 |

## 3.5 Temporary Files and Cleanup
The pipeline generates several temporary Zarr stores during processing. These follow the naming convention:


``` bash
tmp_YYYY-MM-DD_HHhMM_ndvi_01_downloaded_<start>_<end>.zarr   # raw download
tmp_YYYY-MM-DD_HHhMM_ndvi_01_downloaded_<start>_<end>A.zarr  # intermediate step
tmp_*_downloaded_*_processed.zarr                             
``` 

These temporary files are not automatically deleted (cleanup lines are commented out in 0_1_run_pipeline.sh). They can be deleted manually after verifying the pipeline completed successfully:

``` bash

rm -rf /mnt/data2/UniBe-swiss-ndvi/data/tmp_*.zarr
``` 

## 3.6 Configuring Paths for a New Machine

All hard-coded paths are concentrated in the shell scripts (0_1_run_pipeline.sh, 0_1_run_historic_analysis.sh) and at the top of each Python script. When deploying on a new machine:
- 1.	Update VENV_PATH to the new virtual environment location
- 2.	Update HISTO_INPUT to the path of the historical NDVI Zarr
- 3.	Update OUTPUT_ZARR / OUTPUT_TIFF_BASE to the desired output directories
- 4.	Update the forest_mask_bits.zarr path in script 1
- 5.	Update DASK_TEMP_DIR to a path with sufficient scratch space



# 4. Hardware and Storage Requirements

This chapter provides concrete storage estimates, RAM benchmarks, and guidance for scaling the pipeline across different spatial extents.

## 4.1 Dataset Size Estimates

The following estimates are based on the current production dataset (Zarr v3, zstd compression, blosc shuffle):

| Dataset | Pixels | Dates | Approx. Size |
| :--- | :--- | :--- | :--- |
| **Demo (10 km x 10 km patch)** | 4,216 | ~3,200 | 40 MB |
| **Small test (1,000 km²)** | 586,503 | ~3,200 | ~5 GB |
| **Regional (10,000 km²)** | 16,041,205 | ~3,200 | ~40 GB |
| **Full Switzerland (forest)** | 105,715,396 | ~3,200 | ~380 GB |
| **Annual TIFF archive (full CH)** | — | ~70 dates/year | ~50 GB/year |

## 4.2 RAM Requirements by Processing Stage

| Script / Stage | RAM Guidance |
| :--- | :--- |
| **Script 1: Download** | **Low (< 8 GB).** Downloads are processed tile-by-tile and written to Zarr incrementally. |
| **Script 4: Merge** (`4_merge_zarr.py`) | **50 workers x 24 GB = 1.2 TB** for full CH. Reduce `N_WORKERS` for smaller machines. |
| **Script 5: Analysis** (`5_analyse_demo_efficient.py`) | **30 workers x 120 GB = 3.6 TB** for full CH. `PIXEL_CHUNKS=40000` controls memory per worker. |
| **Script 2: Historic processing** (`2_historical_ndvi_test.py`) | **80 workers x 20 GB = 1.6 TB.** Batch processing of 200k pixels keeps peak memory manageable. |
| **Script 6: COG TIFF generation** | **Low (< 16 GB).** Grid reconstruction done one date at a time. |

<div style="background-color: #ebf3fb; border: 1px solid #336699; padding: 10px; font-style: italic; color: #2c3e50;">
  The PIXEL_CHUNKS parameter in scripts 4 and 5 is the primary lever for controlling memory use per Dask worker. For a machine with 32 GB RAM and 8 workers, use PIXEL_CHUNKS=5000 and N_WORKERS=8.
</div>

## 4.3 Runtime Benchmarks

Observed runtimes from the production server (60–120 Dask workers):

| Stage | Pixel Count | Observed Runtime |
| :--- | :--- | :--- |
| **Script 1: Download (1 year)** | 105M | ~2–4 hours (network bound) |
| **Script 4: Merge** | 586K | 90 seconds |
| **Script 4: Merge** | 16M | ~15 minutes |
| **Script 5: Analysis** | 586K | 57 seconds |
| **Script 5: Analysis** | 16M | ~55 minutes |
| **Script 2: Historic (full CH, 8yr)** | 105M | ~6 hours per 200K batch |
| **Script 6: TIFF generation (one date)** | 105M | ~5 minutes |

# 5. End-to-End Quick Start

This chapter provides a minimal, linear walkthrough for running the complete pipeline from scratch. Two modes are described: the continuous update pipeline (daily/weekly operation) and the historical processing pipeline (one-time backfill from 2017).

## 5.1 Continuous Update Pipeline (0_1_run_pipeline.sh)

This is the primary operational mode, intended to be run periodically (e.g. weekly) to ingest new Sentinel-2 images and update the historical NDVI Zarr.

Step 1: Activate the environment and configure dates
``` bash

source /path/to/.venv/bin/activate
# Edit 0_1_run_pipeline.sh: set HISTO_INPUT and END_DATE
``` 

Step 2: Run the pipeline
``` bash

bash 0_1_run_pipeline.sh 2>&1 | tee pipeline_$(date +%Y%m%d).log
``` 

The pipeline executes four scripts in sequence:

| Step | Script | What it does |
| :--- | :--- | :--- |
| **1** | `1_extract_swisstopo_dataset.py <start> <end>` | Queries Swisstopo STAC API, downloads bands, computes NDVI/NDSI, writes raw Zarr. |
| **2** | `4_merge_zarr.py <download.zarr> <historic.zarr>` | Resamples to daily, extends back to last historic date, writes processed Zarr. |
| **3** | `5_analyse_demo_efficient.py <processed.zarr> <historic.zarr>` | Applies outlier detection and LOESS smoothing, appends new dates to historic Zarr. |
| **4** | `6_create_cogtiff.py <start> <end>` | Creates Cloud-Optimised GeoTIFFs for the newly processed dates. |


Step 3: Verify outputs
``` bash

# Check that new dates appear in the historic Zarr
python 0_2_get_last_date.py $HISTO_INPUT

# Check that TIFFs were created
ls -lh /mnt/data1/UniBe-swiss-ndvi/data/tiffs/ | tail -5
``` 

## 5.2 Historical Backfill Pipeline (0_1_run_historic_analysis.sh)

This is a one-time operation to process all observations from April 2017 to November 2025 and build the historical NDVI Zarr from scratch. It runs scripts in this order:

-	(Optionally) 0_create_lookup_table.py — builds the median NDVI lookup table from the double-logistic model parameters. Already done; output at lookup_table_median_ndvi_v7.zarr.
-	1_download_satellite_images.py — downloads all years (parallelised over years using GNU parallel).
-	1b_merge_satellite_image_downloads.py — concatenates the per-year download Zarrs into one file.
-	2_historical_ndvi_test.py — applies the historical processing function in pixel batches and writes the final historic Zarr.
-	(Optionally) 2b_crop_to_2025-11-30.py — crops the result to a specific date if needed.
-	4_plot_historic_tiff.py — generates TIFF files for all or selected dates.

## 5.3 Checking the Start Date Automatically

The pipeline automatically determines the start date for the download from the last date present in the historical Zarr:
``` bash

START_DATE=$(python 0_2_get_last_date.py $HISTO_INPUT)
echo "Last historic date: $START_DATE"
``` 

This ensures the download is always incremental and avoids duplicating already-processed dates.

## 5.4 Running for a Single Date or Date Range

To process a specific date range (e.g., for testing or gap-filling):
``` bash

# Download only
python 1_extract_swisstopo_dataset.py 2025-06-01 2025-06-30

# Full pipeline for a custom date range (edit dates in the shell script)
# Set START_DATE and END_DATE in 0_1_run_pipeline.sh then run:
bash 0_1_run_pipeline.sh

# Generate a TIFF for a single specific date
python 4_plot_historic_tiff.py $HISTO_INPUT 2024-07-22
```

# 6. Entry Points for Common Use Cases

This chapter provides a concise reference for the most common operations a user or operator will need to perform.

## 6.1 Running the Full Pipeline from Scratch

A fresh installation requires three phases:
-	Setup: Install system dependencies, create virtualenv, pip install -r requirements.txt (see Chapter 1).
-	Data: Ensure the forest mask Zarr and lookup table Zarr are present (see Chapter 3).
-	Execute: Run 0_1_run_historic_analysis.sh to backfill all historic data, then schedule 0_1_run_pipeline.sh for ongoing updates.

## 6.2 Updating the Historical NDVI Dataset (Incremental)
For weekly or monthly updates after initial setup:

``` bash

# 1. Verify last date in historical Zarr
python 0_2_get_last_date.py /path/to/ndvi_historic.zarr

# 2. Edit 0_1_run_pipeline.sh: set END_DATE to today or desired end
# 3. Run the pipeline (it auto-detects START_DATE from the Zarr)
bash 0_1_run_pipeline.sh
``` 

## 6.3 Generating TIFFs for a Set of Dates

``` bash

# Single date
python 4_plot_historic_tiff.py /path/to/ndvi_historic.zarr 2024-08-15

# All dates in the Zarr
python 4_plot_historic_tiff.py /path/to/ndvi_historic.zarr all_dates
``` 

## 6.4 Inspecting the Historical Zarr Structure

``` bash


python -c "
import xarray as xr
ds = xr.open_zarr('/path/to/ndvi_historic.zarr')
print(ds)
print('First date:', ds.date.values[0])
print('Last date:', ds.date.values[-1])
print('N pixels:', len(ds.pixel))
"
``` 

## 6.5 Plotting a Time Series for a Specific Location
Use script 3_check_historical_ndvi.py as a starting point. Configure the XY_COORDS dictionary at the top of the script with your location (in EPSG:2056 coordinates) and set PROC_ZARR to the path of your processed Zarr, then run:

``` bash

python 3_check_historical_ndvi.py
``` 

## 6.6 Checking for Errors and Retrying Failed Downloads
Script 1 automatically retries failed time steps once. If retries also fail, check:

-	Network connectivity to data.geo.admin.ch
-	The log file for specific HTTP error codes
-	Re-run the script with the same date range — already-completed Zarr time steps are not re-downloaded


# 7. Configuration and Hard-Coded Parameters

The pipeline currently uses hard-coded parameters spread across the Python scripts and shell scripts. This chapter documents all key parameters, their locations, and recommended values.

## 7.1 Date Parameters

| Parameter | Location and Description |
| :--- | :--- |
| **START_DATE** | `0_1_run_pipeline.sh` — auto-derived from `HISTO_INPUT` using `0_2_get_last_date.py`. Override manually if needed. |
| **END_DATE** | `0_1_run_pipeline.sh` — set to yesterday by default. Hardcode to a specific date for controlled runs. |
| **start_tiff_date / end_tiff_date** | `6_create_cogtiff.py` — computed as fifth-to-last and fourth-to-last observation dates in the range to ensure $\pm3$ observations for L2 smoothing. |

## 7.2 Parallelism and Chunking Parameters

These are the most important performance parameters. Adjust based on available hardware:

| Parameter | Script | Default / Description |
| :--- | :--- | :--- |
| **N_WORKERS** | 4, 5, 2_hist, 4_plot | 30–80 Dask workers. Set to the number of available CPU cores. |
| **MEMORY_PER_WORKER** | 4, 5 | "120GB" per worker. **Total RAM = N_WORKERS x MEMORY_PER_WORKER.** |
| **PIXEL_CHUNKS** | 4, 5 | 40,000 pixels per chunk. Reduce this value if workers run out of memory. |
| **DATE_CHUNKS / OUT** | 4, 5 | 365 dates per chunk in output. Controls I/O and Zarr storage granularity. |
| **BATCH_SIZE** | 2_historical | 5 x `PIXEL_CHUNKS` = 200,000. Controls the outer loop memory footprint. |
| **INNER_PIXEL_CHUNK** | 2_historical | `BATCH_SIZE` / `N_WORKERS` / 20. Fine-tunes Dask task granularity. |

## 7.3 Sentinel-2 Processing Parameters

| Parameter | Location | Value / Description |
| :--- | :--- | :--- |
| **INVALID** | All scripts | `-32768` ($-2^{15}$). Sentinel-2 pixels masked as cloud shadow. |
| **NO_COVERAGE** | All scripts | `32767` ($2^{15}-1$). Pixels with no data for a given time step (no overpass or snow-masked). |
| **Snow mask threshold (NDSI)** | `1_download_...` | `NDSI >= 0.43 x 10000`. Pixels above this NDSI are treated as snow-covered and set to `NO_COVERAGE`. |
| **Aggregation method** | `4_merge_zarr.py` | `'first'` — keeps the first observation when multiple overpasses occur on the same day. |

## 7.4 Outlier Detection Parameters

The outlier detection logic in both the continuous (5_analyse_demo_efficient.py) and historic (2_historical_ndvi_test.py) processing functions uses:

| Parameter | Value / Description |
| :--- | :--- |
| **`delta_threshold`** | **0.1 (continuous) / 0.05 (historic).** Minimum absolute $\Delta$-NDVI (relative to seasonal median) for a point to be considered a potential outlier. |
| **`delta_delta_threshold`** | **0.1.** Minimum change in $\Delta$-NDVI between consecutive observations. Both thresholds must be exceeded to confirm an outlier. |
| **Boundary conditions** | **NDVI < 0.05 or > 0.95**: Smoothing is bypassed and raw delta is kept. This prevents smoothing across extreme events (e.g., fire, senescence). |
| **Extreme negative threshold** | **Sum of $\Delta < -0.2$ in a 7-observation window ($\text{count} \ge 5$)**: Smoothing is bypassed to specifically handle fire events. |

## 7.5 Compression Parameters

| Setting | Value |
| :--- | :--- |
| **Compressor** | `Blosc` / `zstd`, `clevel=3`, `shuffle=bitshuffle` (for `int16` data) or `byteshuffle` (for `float`) |
| **Zarr format** | Version 3 throughout |
| **TIFF compression** | `deflate` (COG driver via `rioxarray`/`GDAL`) |
| **TIFF dtype** | `int16` (NDVI × 10,000; range -10,000 to 10,000) |


# 8. Data Format and Schema Documentation

This chapter documents the complete schema of all key data structures produced and consumed by the pipeline

## 8.1 Historical NDVI Zarr (ndvi_historic_vN.zarr)

This is the primary output and the main input to the continuous update pipeline. It is a Zarr v3 store with the following structure

| Field | Description |
| :--- | :--- |
| **Dimensions** | `pixel` (N ≈ 105,715,396 for full CH), `date` (3200+ daily dates from 2017-04-03) |
| **Coordinate: pixel** | `int32`. Pixel ID from 0 to N-1, mapping to forest pixel positions in the reference grid. |
| **Coordinate: date** | `datetime64[ns]`. Daily dates (no gaps). |
| **Coordinate: doy** | (`date`). `int32`. Day of year (1–365, leap days mapped to 365). |
| **Coordinate: x** | (`pixel`). `int32`. Easting in EPSG:2056 (meters), center of pixel. |
| **Coordinate: y** | (`pixel`). `int32`. Northing in EPSG:2056 (meters), center of pixel. |
| **Coordinate: x_idx** | (`pixel`). `int32`. Row index in the 24,542 x 37,728 reference grid. |
| **Coordinate: y_idx** | (`pixel`). `int32`. Column index in the reference grid. |
| **Data variable: ndvi_processed** | (`pixel`, `date`). `int16`. Processed NDVI × 10,000. Range: -10,000 to 10,000. `NO_COVERAGE` = 32767, `INVALID` = -32768. |
| **Data variable: mask_array** | (`pixel`, `date`). `int8`. Processing status (see mask legend below). |

## 8.2 mask_array Legend

The mask_array variable encodes the processing status of each pixel-date combination:

| Value | Meaning |
| :--- | :--- |
| **0** | **Not an observation date, not yet smoothed** (gap between observations, before L2 processing) |
| **1** | **Not an observation date, smoothed** (gap-filled by LOESS interpolation, L2 finalized) |
| **2** | **Observation date, not yet smoothed** (raw satellite observation, pending L2) |
| **3** | **Observation date, smoothed** (satellite observation used in L2 smoothing, finalized) |
| **4** | **Observation date, flagged as outlier** (excluded from smoothing, replaced by interpolation) |

<div style="background-color: #ebf3fb; border: 1px solid #336699; padding: 10px; font-style: italic; color: #2c3e50;">
  In the continuous update pipeline, only dates with mask_array = 3 or 1 should be considered 'finalized' L2 products. Values of 2 and 0 in the most recent dates indicate data that has not yet been smoothed due to the latency window (requires observations 3 before and 3 after).
</div>

## 8.3 Raw Download Zarr (tmp_*_downloaded_*.zarr)

| Field | Description |
| :--- | :--- |
| **Dimensions** | `datetime` (T observations), `pixel` (N forest pixels) |
| **`datetime`** | `datetime64[ns]`. Sub-daily observation timestamps (UTC). |
| **`ndvi`** | `int16`. Raw NDVI × 10,000. `NO_COVERAGE` or `INVALID` for missing/masked pixels. |
| **`ndsi`** | `int16`. Raw NDSI × 10,000. |
| **`x`, `y`, `x_idx`, `y_idx`** | Pixel coordinates (consistent with historic Zarr). |
| **Attribute: `nodata`** | `32767` (`NO_COVERAGE`) |
| **Attribute: `cloud_shadow`** | `-32768` (`INVALID`) |

## 8.4 COG GeoTIFF Output

| Property | Value |
| :--- | :--- |
| **Driver** | `COG` (Cloud-Optimised GeoTIFF) |
| **Compression** | `DEFLATE` |
| **Data type** | `int16` |
| **CRS** | `EPSG:2056` (CH1903+ / LV95) |
| **Pixel size** | 10 m x 10 m |
| **Extent** | Compact window covering all forest pixels (not full CH bbox) |
| **Naming convention (NDVI)** | `YYYYMMDD.tiff` or `YYYYMMDD_historic.tiff` |
| **Naming convention (mask)** | `YYYYMMDD_mask.tiff` or `YYYYMMDD_historic_mask.tiff` |
| **NO_DATA value** | `NaN` (float stored as `int16` — often rendered as `0` in viewers; use `rasterio` to read correctly) |

## 8.5 Lookup Table Zarr (lookup_table_median_ndvi_v7.zarr)

| Field | Description |
| :--- | :--- |
| **Dimensions** | `pixel` (N forest pixels), `doy` (365 days of year) |
| **`median_ndvi`** | `int16`. Expected median NDVI × 10,000 for each pixel and day-of-year, derived from the double-logistic seasonal model (lower + upper envelope average). |
| **Use** | Joined to the processed dataset by `(pixel, doy)` to compute $\Delta\text{-NDVI} = \text{observed} - \text{expected}$, which drives outlier detection and smoothing. |

# 9. Processing Methods

This chapter provides a detailed description of the NDVI processing algorithm, covering L1 and L2 product definitions, outlier detection, and the LOESS smoothing approach.

## 9.1 Overview and Product Levels

The pipeline produces two conceptual product levels:

-	L1 (not yet implemented as a separate output): raw observations with provisional outlier flags. No smoothing. Suitable for near-real-time monitoring but subject to revision.
-	L2 (current output): definitive outlier removal followed by LOESS gap-filling and smoothing. Has a processing latency of approximately 3–4 observation dates (roughly 10–20 days). All mask_array = 1 and 3 values are finalized L2.

## 9.2 Seasonal Reference: The Median NDVI Model

All processing is relative to an expected seasonal cycle. For each pixel and each day of year, a median NDVI value is pre-computed from the double-logistic phenology model fitted to the full 2017–2025 observation record. This is stored in the lookup table Zarr.

The delta-NDVI is defined as: delta = observed_NDVI - median_NDVI. Processing (outlier detection, smoothing) operates on delta-NDVI rather than raw NDVI, making the algorithm insensitive to the baseline seasonal amplitude.

## 9.3 Outlier Detection

An observation is flagged as an outlier if all three of the following conditions are met simultaneously:

-	The absolute delta-NDVI of the point exceeds delta_threshold (0.05 in historic processing, 0.1 in continuous processing).
-	The absolute change in delta-NDVI between the preceding observation and the point exceeds delta_delta_threshold (0.1).
-	The absolute change in delta-NDVI between the point and the following observation also exceeds delta_delta_threshold.

This requires that an outlier be both anomalous relative to the seasonal baseline AND isolated (both neighbors show a large jump back toward normal). This design avoids flagging sustained anomalies (e.g., a real drought) as outliers, while catching single-observation spikes from residual clouds.

## 9.4 LOESS Smoothing

After outlier removal, the delta-NDVI sequence is smoothed using LOESS (locally weighted regression) via statsmodels.nonparametric.lowess with the following settings:

-	frac = 7 / len(delta_ndvi_2): adaptive bandwidth, equivalent to a 7-observation window. This provides approximately 20–35 day influence depending on overpass frequency.
-	it = 3: three iterations of robust re-weighting to down-weight residual anomalies.
-	return_sorted = False: evaluates LOESS at the original x positions.

Boundary and extreme-value exceptions to smoothing:
-	If any observation in a 7-point window has NDVI < 0.05 or > 0.95 (near saturation), the raw delta is retained for the center point.
-	If 5 or more of the 7 observations have delta < -0.2 (large sustained negative anomaly, e.g. fire), the raw delta is retained.

## 9.5 Gap-filling by Linear Interpolation (L1)

After LOESS smoothing of the delta-NDVI at observation dates, the delta is linearly interpolated to all daily time steps using numpy.interp. The smoothed NDVI is then reconstructed as:

``` bash

ndvi_smoothed = 10000 * (interpolated_delta + median_ndvi)
``` 

The interpolation anchors at zero delta at the first and last dates, ensuring the smoothed curve does not diverge beyond the seasonal expectation at the boundaries of the record.

## 9.6 Processing Latency and L2 Finalization

L2 finalization requires at least 3 observations after a given date (to ensure the LOESS window can look forward). In the continuous update pipeline:

-	Dates with mask_array = 0 or 2: raw or not yet smoothed — subject to revision in the next pipeline run.
-	Dates with mask_array = 1 or 3: finalized L2 — will not change in subsequent runs.
-	TIFF generation in script 6 is triggered for the fifth-to-last to fourth-to-last observation dates, ensuring finalization before output.

Expected latency: approximately 10–20 calendar days after acquisition, depending on cloud cover and overpass frequency.



# 10. Validation and Error Handling

This chapter documents the validation checks built into the pipeline and guidance for diagnosing common errors.

## 10.1 Built-in Validation in Script 5

Before appending to the historical Zarr (when HISTO_ZARR_OUTPUT == HISTO_ZARR_INPUT), script 5 performs the following structural checks:

-	Dimensions of the dataset to append match the existing Zarr dimensions.
-	Coordinates (pixel, x, y, x_idx, y_idx) are identical in shape, dtype, and values.
-	Data variable dtypes (ndvi_processed, mask_array) match.
-	After appending, the boundary dates are exactly 1 day apart (no gaps or duplicates introduced).

If any check fails, the append is aborted and the script writes the full merged dataset to a fallback file (.failedAppending_YYYYMMDDHHMMSS) rather than corrupting the main Zarr.

## 10.2 Common Errors and Solutions

| Error | Cause | Solution |
| :--- | :--- | :--- |
| **`OSError: Too many open files`** | Zarr opens one file per chunk; default OS limit is too low. | Run: `ulimit -n 8192` before executing the script. |
| **Dates not exactly 1 day apart** | Zarr was appended with duplicate or missing dates at the interface. | Check `START_DATE` in `0_1_run_pipeline.sh`; the fallback file will contain correct data. |
| **`PerformanceWarning: Chunks`** | Rechunking during processing. | **Expected; ignore.** This is caused by mismatched chunk sizes between historic and new data. |
| **No data downloaded** | No Sentinel-2 observations in the requested date range. | Check the Swisstopo STAC API manually; there may be a gap in the archive. |
| **`NaT` in datetime coordinate** | Occasional null timestamps in the download stream. | Script 1b handles this via a `preprocess()` function that drops `NaT` values. |
| **Zarr format mismatch (v2 vs v3)** | Trying to open a v3 Zarr store with a v2 reader. | Always pass `zarr_format=3` to `to_zarr()`; open with `xr.open_zarr()`. |

## 10.3 Versioning and Reproducibility

The pipeline does not yet have a formal versioning system for data–code compatibility. The following conventions are used in practice:

-	The historical Zarr filename encodes the processing date and version: historical_2026-04-04_18h16_historical_v7.zarr
-	The v7 label corresponds to a specific code commit and processing parameter set. New processing runs should increment the version label.
-	The shell script 0_1_run_historic_analysis.sh logs the git commit hash at runtime, providing a record of which code version was used.
-	The lookup table Zarr version (v7) must match the historical Zarr version — they share the same pixel ordering and DOY definition.


# 11. Script-by-Script Reference

This chapter provides a concise reference for each script in the pipeline, documenting inputs, outputs, configurable parameters, and known limitations

## 11.1 0_2_get_last_date.py

Purpose: prints the last date present in a historical NDVI Zarr to stdout. Used by the pipeline shell script to determine the download start date.

-	Input: path to historical Zarr (command-line argument)
-	Output: YYYY-MM-DD string to stdout
-	Configurable parameters: none

## 11.2 1_extract_swisstopo_dataset.py / 1_download_satellite_images.py

Purpose: queries the Swisstopo STAC API for Sentinel-2 observations in the specified date range, downloads band data, computes NDVI and NDSI for all forest pixels, and writes the result to a Zarr store.

-	Inputs: start_date, end_date (CLI arguments)
-	Outputs: tmp_*_downloaded_*.zarr
-	Key parameters: OUTPUT_ZARR path (top of script), PIXEL_CHUNKS=10000
-	Known issue: the 'current' mosaic item must be filtered out (handled automatically)
-	Note: 1_download_satellite_images.py is the newer version and adds a snow mask (NDSI >= 0.43) absent in 1_extract_swisstopo_dataset.py

## 11.3 1b_merge_satellite_image_downloads.py

Purpose: concatenates multiple per-year download Zarr files into a single multi-year Zarr. Used only for the historical backfill pipeline.

-	Inputs: list of per-year zarr paths (hard-coded in the script)
-	Outputs: a single merged Zarr covering all years
-	Note: removes NaT timestamps automatically via the preprocess() function

## 11.4 4_merge_zarr.py

Purpose: converts the raw sub-daily download Zarr to a daily time series, fills gaps back to the last historic date with NO_COVERAGE, and prepares the dataset for analysis by script 5.

-	Inputs: download Zarr path, historic Zarr path (CLI arguments)
-	Outputs: *_processed.zarr (path derived from input by adding _processed)
-	Key parameters: N_WORKERS=50, MEMORY_PER_WORKER='24GB', PIXEL_CHUNKS=40000, DATE_CHUNKS=365
-	Aggregation method: 'first' (takes the first overpass when multiple occur on the same day)
-	Note: the question of whether to use 'mean' instead is still open (see TODO comments)

## 11.5 5_analyse_demo_efficient.py

Purpose: applies outlier detection and LOESS smoothing to the merged dataset and appends the newly processed dates to the historical Zarr.

-	Inputs: processed Zarr path, historic Zarr path, optional output path (CLI arguments)
-	Outputs: extended historic Zarr (appended in-place or written to new path)
-	Key parameters: N_WORKERS=30, PIXEL_CHUNKS=40000, DATE_CHUNKS_OUT=365
-	Append vs rewrite: append is attempted first; falls back to full rewrite on failure
-	Known limitation: the starting_date logic (3 prior observations) means the very beginning of the historic record cannot be fully smoothed without at least 3 prior observations

## 11.6 6_create_cogtiff.py

Purpose: generates Cloud-Optimised GeoTIFFs from the processed Zarr for the newly completed dates (fifth-to-last to fourth-to-last observation dates in the requested range).
-	Inputs: start_date, end_date (CLI arguments); reads from INPUT_BASE Zarr (hard-coded)
-	Outputs: YYYYMMDD.tiff and YYYYMMDD_mask.tiff in OUTPUT_TIFF_BASE
-	Skips already-exported dates automatically
-	Note: the -100 value mentioned in the README feedback refers to the difference between NO_COVERAGE (32767) and the int16 range; in practice NaN is used for missing pixels in the grid reconstruction

## 11.7 4_plot_historic_tiff.py

Purpose: generates TIFFs from the historical Zarr for any requested set of dates. Supports single dates or all_dates mode.
-	Inputs: historic Zarr path, date string or 'all_dates' (CLI arguments)
-	Outputs: YYYYMMDD_historic.tiff and YYYYMMDD_historic_mask.tiff
-	Uses a compact window transform derived from the actual min/max pixel coordinates, producing smaller TIFFs than the full CH bounding box

## 11.8 2_historical_ndvi_test.py
Purpose: applies the historical processing function (LOESS smoothing of delta-NDVI) to the full 2017–2025 observation record in pixel batches. This is the one-time historical backfill script.
-	Inputs: download Zarr, lookup table Zarr (hard-coded at top of script)
-	Outputs: historical_*.zarr
-	Key parameters: BATCH_SIZE=5xPIXEL_CHUNKS, INNER_PIXEL_CHUNK=BATCH_SIZE/N_WORKERS/20
-	Processing: batches are processed sequentially, with each batch persisted to Dask workers before applying_ufunc to avoid memory fragmentation
 
